In [ ]:
INF = float('inf')
def myprint(arr):
    row = -1
    for lst in arr:
        row+=1
        if row == 0: continue
        print(*lst[1:N+1], sep=' ')

def update_arr(ri,rj,slst):
    arr = [[0]*(N+1) for _ in range(N+1)]
    arr[ri][rj] = -1
    for sn,si,sj,ss,sp,so in slst:
        if so == 1 or not 1<=si<=N or not 1<=sj<=N: continue
        arr[si][sj] = sn
    return arr

def cal_dist(si,sj,ei,ej):
    return (si-ei)**2 + (sj-ej)**2

def move_r(ri,rj,slst):
    # 가장 가까운 산타 선택
    cand = []
    for sn,si,sj,ss,sp,so in slst:
        # 탈락한 산타는 건너뛰기
        if so == 1: continue
        dist = cal_dist(ri,rj,si,sj)
        cand.append([dist,si,sj])
    cand.sort(key=lambda x: (x[0],-x[1],-x[2]))
    # print(cand)

    # 가장 가까운 이동 방향 선택
    min_dist = INF
    min_dist_i = 0
    for i in range(8):
        dist = cal_dist(ri+di[i],rj+dj[i],cand[0][1],cand[0][2])
        if min_dist > dist:
            min_dist = dist
            min_dist_i = i
    
    return ri+di[min_dist_i], rj+dj[min_dist_i],min_dist_i

##################

N,M,P,C,D = map(int,input().split())
ri,rj = map(int,input().split())

# 산타 번호, i, j, 점수, 기절여부, 탈락여부
slst = [list(map(int,input().split())) + [0]*3 for _ in range(P)]
slst.sort(key=lambda x: x[0])

# -1 루돌프, 번호 산타
arr = update_arr(ri,rj,slst)

# 상우하좌
di = [-1,0,1,0,-1,-1,1,1]
dj = [0,1,0,-1,-1,1,-1,1]

# M개 턴에 걸쳐 진행
for m in range(M):

    # 2. 루돌프 움직임
    # 2.1. 탈락하지 않은 산타 중 가장 가까운 산타 선택
    # 2.2. 거리 중 i -> j 큰 산타 선택
    # 2.3. 8방향 중 가장 가까워지는 방향으로 한 칸 이동
    ri,rj,dir = move_r(ri,rj,slst)
    arr = update_arr(ri,rj,slst)
    # print(ri,rj)

    # 2.4. 루돌프가 움직이고 산타와 충돌시 해당 이동 방향으로 산타 C칸 밀림 + 산타 C점 획득, 산타 기절
    # 밀린 위치가 범위 밖이면 탈락
    # 밀린 위치에 산타 있으면 상호작용
    for sn,si,sj,ss,sp,so in slst:
        # 산타와 충돌시 C칸 밀리고 C점 획득, 산타 기절
        if (ri,rj) == (si,sj):
            slst[sn-1][1] += di[dir] * C
            slst[sn-1][2] += dj[dir] * C
            slst[sn-1][3] += C
            slst[sn-1][4] = 1
            arr = update_arr(ri,rj,slst)
        else: continue # 루돌프와 만난 산타 아니면 건너뛰기
        
        # 상호작용
        # 밀린 위치에 산타 있으면 해당 칸부터 빈 공간이 나올 때까지 group에 추가하고 해당 group 내 산타 모두 1칸 이동
        cn,ci,cj = sn,slst[sn-1][1],slst[-1][2]
        group = []
        for sn,si,sj,ss,sp,so in slst:
            # 내가 아닌데 같은 위치에 있는 산타에 대해서
            if cn != sn and (ci,cj) == (si,sj):
                # print("상호작용",(ci,cj),(si,sj),cn,sn)
                while True:
                    ci+=di[dir]
                    cj+=dj[dir]
                    # 격자 밖 또는 빈공간이면 멈추기
                    if not 1<=ci<=N or not 1<=cj<=N or arr[ci][cj] == 0: break
                    else: group.append(arr[ci][cj])
                break
        if group:
            # print("그룹",group)
            for idx in group:
                slst[idx][1] += di[dir]
                slst[idx][2] += di[dir]

    # 범위 밖이면 탈락
    for sn,si,sj,ss,sp,so in slst:
        if not 1<= slst[sn-1][1] <= N or not 1<=slst[sn-1][2] <= N:
            slst[sn-1][5] = 1
    update_arr(ri,rj,slst)
        
    print("루돌프 움직인 후")
    print(ri,rj)
    print(slst)
    myprint(arr)

    # 3. 산타 움직임
    # 3.1. 번호 순 움직임
    # 3.2. 기절했거나 탈락한 산타 못 움직임. 기절했으면 넘기고 기절 풀기
    # 3.3. 루돌프와의 거리가 가장 가까워지는 방향 상우하좌 순으로 1칸 이동
    # 3.4. 다른 산타 칸, 격자 밖 못 움직임
    # 3.5. 가까워질 방법 없으면 안 움직임
    for sn,si,sj,ss,sp,so in slst:
        if so == 1: continue # 탈락 산타 넘어가기
        # 기절했으면 기절 풀고 넘어가기
        if sp == 1:
            slst[sn-1][4] = 0
            continue
        
        # 상우하좌에 대해서 가장 가까운 방향 고르기
        d_min = INF
        d_min_i = 0
        cdist = cal_dist(ri,rj,si,sj)
        for i in range(4):
            ni,nj = si+di[i],sj+dj[i]
            # 다른 산타가 있거나 격자 밖이면 못 움직임
            if not 1<= ni <= N or not 1<=nj <= N or arr[ni][nj] != 0 :
                continue

            # 거리가 멀어지면 못 움직임
            dist = cal_dist(ri,rj,ni,nj)
            if cdist <= dist: continue

            if d_min > dist:
                d_min = dist
                di_min_i = i
        
        # 산타 이동
        if d_min != INF:
            print("산타 이동 가능", sn,d_min,di_min_i)
            slst[sn-1][1] += di[d_min_i]
            slst[sn-1][2] += dj[d_min_i]
            

            # 3.6. 움직인 곳에 루돌프 있으면 이동 반대 방향으로 D칸 이동 + D점 획득, 산타 기절
            if (slst[sn-1][1],slst[sn-1][2]) == (ri,rj):
                slst[sn-1][1] -= di[d_min_i] * D
                slst[sn-1][2] -= dj[d_min_i] * D
                slst[sn-1][3] += D
                slst[sn-1][4] = 1
            
            update_arr(ri,rj,slst)
            # 3.7. 충돌 후 상호작용

            # 상호작용
            # 밀린 위치에 산타 있으면 해당 칸부터 빈 공간이 나올 때까지 group에 추가하고 해당 group 내 산타 모두 1칸 이동
            cn,ci,cj = sn,slst[sn-1][1],slst[-1][2]
            group = []
            for sn,si,sj,ss,sp,so in slst:
                # 내가 아닌데 같은 위치에 있는 산타에 대해서
                if cn != sn and (ci,cj) == (si,sj):
                    # print("상호작용",(ci,cj),(si,sj),cn,sn)
                    while True:
                        ci+=di[dir]
                        cj+=dj[dir]
                        # 격자 밖 또는 빈공간이면 멈추기
                        if not 1<=ci<=N or not 1<=cj<=N or arr[ci][cj] == 0: break
                        else: group.append(arr[ci][cj])
                    break
            if group:
                # print("그룹",group)
                for idx in group:
                    slst[idx][1] += di[dir]
                    slst[idx][2] += di[dir]

        # 범위 밖이면 탈락
        for sn,si,sj,ss,sp,so in slst:
            if not 1<= slst[sn-1][1] <= N or not 1<=slst[sn-1][2] <= N:
                slst[sn-1][5] = 1
    update_arr(ri,rj,slst)
    
    print("산타 움직인 후")
    print(ri,rj)
    print(slst)
    myprint(arr)



    # 7. 게임 종료
    # 모든 산타 탈락시 게임 종료
    n_out = 0
    for sn,si,sj,ss,sp,so in slst:
        n_out += so
    if n_out == P: break

    # 탈락하지 않은 산타 점수 1점 추가 부여
    for sn,si,sj,ss,sp,so in slst:
        if so == 0:
            slst[sn-1][4] +=1

answer = []
for sn,si,sj,ss,sp,so in slst:
    answer.append(sp)

print(*answer,sep=' ')